# G1 Academy Bonus - Task 3: developing say() and set_headlight() from scratch

## Introduction
Building on Task 2's DDS/pub-sub helpers, this task develops full native versions of `say` and `set_headlight`, matching the behavior of `sdk_wrapper.G1.say`/`set_headlight` (not just the shortcut used in the intro academy notebook). We still reuse `util.py`'s Piper subprocess/WAV-conversion boilerplate - that part is intentionally not something to retype - but we build the `AudioClient` plumbing, color parsing, and the cancellable background headlight thread ourselves.

## Task 1 - `AudioClient` + `say()`
`AudioClient` is the native SDK request client for volume, `LedControl`, and `PlayStream`. `util.play_piper_text` does only the tedious part (Piper synthesis, resampling to mono/16-bit/16kHz, `PlayStream` framing); you own constructing and initializing the client.

In [ ]:
import sys
import time
sys.path.append("..")
from unitree_sdk2py.g1.audio.g1_audio_client import AudioClient
from util import play_piper_text
from sdk_wrapper import ensure_channel_factory

ensure_channel_factory(0, "eth0")

# TODO: construct, set timeout, and initialise one AudioClient.
audio_client = None

def say(text, language="en", volume=100):
    # TODO: call the supplied Piper/PlayStream helper with the initialised client.
    raise NotImplementedError

print(say("Headlight and speech helpers online."))


## Task 2 - Named/hex/rgb color parsing + intensity scaling
`set_headlight(color=...)` accepts a name, a `#RRGGBB` hex string, or an `"R,G,B"` string. Parse each form into a clamped `(r, g, b)` tuple, then scale it by an intensity percentage before it is ever sent to `LedControl`.

In [ ]:
import re

_NAMED_COLORS = {
    "white": (255, 255, 255), "red": (255, 0, 0), "green": (0, 255, 0), "blue": (0, 0, 255),
    "yellow": (255, 255, 0), "cyan": (0, 255, 255), "magenta": (255, 0, 255),
    "orange": (255, 165, 0), "purple": (128, 0, 128), "pink": (255, 105, 180),
}

def parse_color(value):
    # TODO: support a named color, #RRGGBB, and R,G,B; clamp each channel to 0..255.
    raise NotImplementedError

def scale_color(rgb, intensity):
    # TODO: clamp intensity to 0..100 and scale the RGB tuple.
    raise NotImplementedError

parse_color("cyan"), parse_color("#00ffff"), parse_color("0,255,255")


## Task 3 - Cancellable background headlight thread + `set_headlight()`
`LedControl` only sets the color for a moment, so holding a color for `duration_s` needs a thread that refreshes it periodically. The thread must be safely cancellable: a new call to `set_headlight` (a new color, or a mode/controller transition) must stop and join any earlier thread before starting its own, and the thread must always attempt to turn the light off in its `finally` block so a crash does not leave a color stuck on.

In [ ]:
import threading
import time

def led_control_was_accepted(code):
    # The G1 LED RPC can time out (3104) even when the command was applied.
    return int(code) in (0, 3104)

class HeadlightThread(threading.Thread):
    # TODO: retain the supplied parameters, refresh LedControl until stopped/expired,
    # and always attempt to turn the LED off in finally.
    pass

_headlight_stop = None
_headlight_thread = None

def set_headlight(color="green", intensity=100, duration_s=3):
    # TODO: issue one immediate LED command; cancel/join a previous worker before
    # starting a new one for positive duration. Treat return codes 0 and 3104 as accepted.
    raise NotImplementedError

set_headlight("yellow", intensity=100, duration_s=30)


You have now reconstructed `sdk_wrapper.G1.say` and `sdk_wrapper.G1.set_headlight` natively - compare this cell against `_HeadlightThread`/`G1.set_headlight` in `sdk_wrapper.py`.

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.